In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import numpy as np
from sklearn.pipeline import Pipeline


In [5]:
df = pd.read_csv('Processed after encoding')
df.head()

,Unnamed: 0,Gender?,When you go to sleep?,The average time you spend playing games?,How many time you spend with family and friend?,How you fill when you can not play game in whole day?,How you fill to complete game level?,If you didn't finish games last level what is your feeling?,Do you fill Fatigue?,Do you play games for stress relief?,addicted,Which type of game you addict more?_Battle Royale,Which type of game you addict more?_Open World,Which type of game you addict more?_Shooter,Which type of game you addict more?_Sports,Which type of game you addict more?_Strategy
0,0,0,2,4,3,0,0,0,0,1,1,0,0,0,1,0
1,1,0,1,1,15,2,0,0,1,0,0,0,0,1,0,0
2,4,0,1,2,8,2,0,0,0,0,0,0,0,0,1,0
3,5,0,2,5,1,0,0,0,0,1,1,1,0,0,0,0
4,6,0,1,3,5,0,0,0,0,1,0,0,0,0,1,0


In [6]:
X=df.drop(columns='addicted',axis=1)
y=df['addicted']


In [7]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)


In [ ]:

logi = LogisticRegression()
logi.fit(X_train,y_train)
logi_pred=logi.predict(X_test)
print(classification_report(y_test,logi_pred))
print(confusion_matrix(y_test,logi_pred))
print(np.mean(cross_val_score(estimator=logi,X=X,y=y,cv=25)))
print('Train',accuracy_score(y_train,logi.predict(X_train)))
print('CV Score',np.mean(cross_val_score(estimator=logi,X=X,y=y,cv=25)))
param_grid = [{'penalty':['l1','l2','none'],
    'C' : np.logspace(-4,4,20),
    'solver': ['lbfgs','newton-cg','liblinear','sag','saga'],
    'max_iter'  : [100,1000,2500,5000]
}]

clf = GridSearchCV(logi,param_grid = param_grid, cv = 3, verbose=True,n_jobs=-1)
clf.fit(X_train,y_train)

In [9]:
best_logi=clf.best_estimator_
best_logi_par = clf.best_params_

In [10]:
dt = DecisionTreeClassifier(random_state=42)
param_grid1= {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3,5,7,10,None],
    'min_samples_split': [20,50,10],
    'min_samples_leaf': [10,20,40],
    'max_features': [None, 'sqrt', 'log2']
}

grid1 = GridSearchCV(
    estimator=dt,
    param_grid=param_grid1,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

grid1.fit(X_train, y_train)
best_dt = grid1.best_estimator_
print("Decision Tree best parameter",best_dt)
dt_pred = best_dt.predict(X_test)
print(classification_report(y_test,dt_pred))
print(confusion_matrix(y_test,dt_pred))
print(np.mean(cross_val_score(estimator=best_dt,X=X,y=y,cv=25)))
print('Train',best_dt.score(X_train,y_train))
print('Test',best_dt.score(X_test,y_test))
best_dt_par = grid1.best_params_
best_dt_par

Decision Tree best parameter DecisionTreeClassifier(criterion='entropy', max_depth=7, min_samples_leaf=10,
                       min_samples_split=20, random_state=42)
              precision    recall  f1-score   support

           0       0.96      0.95      0.96       145
           1       0.89      0.90      0.89        60

    accuracy                           0.94       205
   macro avg       0.92      0.93      0.92       205
weighted avg       0.94      0.94      0.94       205

[[138   7]
 [  6  54]]
0.9540243902439025
Train 0.9681372549019608
Test 0.9365853658536586


{'criterion': 'entropy',
 'max_depth': 7,
 'max_features': None,
 'min_samples_leaf': 10,
 'min_samples_split': 20}

In [11]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid2= {
    'knn__n_neighbors': list(range(1, 31)),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}
grid2= GridSearchCV(pipe, param_grid2, cv=10, scoring='accuracy', n_jobs=-1)
grid2.fit(X_train, y_train)
bestknn=grid2.best_estimator_
best_knn_par=grid2.best_estimator_
knn_pred= bestknn.predict(X_test)
print(f"Best Parameters: {grid2.best_params_}")
print(f"Best Score: {grid2.best_score_}")
print('Train',bestknn.score(X_train,y_train))
print('Test',bestknn.score(X_test,y_test))
print('CV Score',np.mean(cross_val_score(estimator=bestknn,X=X,y=y,cv=25,scoring='accuracy')))


Best Parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 5, 'knn__weights': 'distance'}
Best Score: 0.9522734116230052
Train 1.0
Test 0.9414634146341463
CV Score 0.9765121951219513


In [12]:
param_grid3 = {'var_smoothing': np.logspace(0, -9, num=100)}
nbModel_grid = GridSearchCV(estimator=GaussianNB(), 
                            param_grid=param_grid3, 
                            verbose=1, 
                            cv=10, 
                            n_jobs=-1)
nbModel_grid.fit(X_train, y_train)
bestnb = nbModel_grid.best_estimator_
bestnb_par=nbModel_grid.best_params_
print(nbModel_grid.best_estimator_)
print(nbModel_grid.best_score_)
print('Train',bestnb.score(X_train,y_train))
print('Test',bestnb.score(X_test,y_test))
print('CV Score',np.mean(cross_val_score(estimator=bestnb,X=X,y=y,cv=25)))



Fitting 10 folds for each of 100 candidates, totalling 1000 fits
GaussianNB(var_smoothing=np.float64(3.5111917342151277e-07))
0.873803071364047
Train 0.8848039215686274
Test 0.9365853658536586
CV Score 0.8805853658536585


In [13]:
results = []

def add_result(model_name, model, best_params, X_train, X_test, y_train, y_test):
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Accuracy
    train_acc = roc_auc_score(y_train, y_train_pred)
    test_acc = roc_auc_score(y_test, y_test_pred)
    cros_val = np.mean(cross_val_score(estimator=model,X=X,y=y,scoring='roc_auc',cv=10))
    
    # Classification report
    report = classification_report(y_test, y_test_pred, output_dict=True)
    
    results.append({
        'Model': model_name,
        'Best Params': best_params,
        #'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Cross-Val Score':cros_val,
        'Precision': report['1']['precision'],
        'Recall': report['1']['recall'],
        'F1-score': report['1']['f1-score'],
        'Overfit Gap': train_acc - test_acc
    })

In [16]:
add_result("Logistic Regression", best_logi, best_logi_par, X_train, X_test, y_train, y_test)

add_result("Decision Tree", best_dt, best_dt_par, X_train, X_test, y_train, y_test)

add_result("KNN", bestknn, best_knn_par, X_train, X_test, y_train, y_test)

add_result("Naive Bayes", bestnb, bestnb_par, X_train, X_test, y_train, y_test)




In [17]:
df_results = pd.DataFrame(results)

df_results = df_results.round(3)

# sort by best model
df_results = df_results.sort_values(by='Test Accuracy', ascending=False)

df_results

,Model,Best Params,Test Accuracy,Cross-Val Score,Precision,Recall,F1-score,Overfit Gap
0,Logistic Regression,"{'C': 0.23357214690901212, 'max_iter': 100, 'p...",0.946,0.961,0.903,0.933,0.918,-0.089
2,KNN,"(StandardScaler(), KNeighborsClassifier(metric...",0.934,0.976,0.887,0.917,0.902,0.066
3,Naive Bayes,{'var_smoothing': 3.5111917342151277e-07},0.931,0.948,0.873,0.917,0.894,-0.054
1,Decision Tree,"{'criterion': 'entropy', 'max_depth': 7, 'max_...",0.926,0.987,0.885,0.900,0.893,0.028
